# Restaurant Franchise Profits – Linear Regression (One Variable) – Extended Solution

**Domain:** Business Analytics / Machine Learning Foundations  
**Inspired by:** Coursera Machine Learning Specialization C1_W2 (Andrew Ng) + audience-adaptation principles

This notebook contains complete, runnable solutions, alternate implementations, extra practice, a Monte-Carlo / hyper-parameter simulation, and audience-adaptation notes.


## Goals
In this extended lab you will:
- Load and explore the classic restaurant-franchise data set (city population → monthly profit).
- Implement the **cost function** \(J(w,b)\) from first principles (loop + vectorized alternate).
- Implement **batch gradient descent** to learn the optimal parameters \((w,b)\).
- Visualize the fitted line, cost history, and prediction for candidate cities.
- Explore **alternate implementations**, additional practice questions, and a **parameterised simulation**.
- Reflect on how to **adapt the same analysis** for different audiences (CEO / executive vs. data scientist vs. mixed technical audience) following the attached audience-analysis guidelines.


## Notation
| Notation | Description | Python |
|:---------|:------------|:-------|
| \(x^{(i)}\) | \(i\)-th city population (in 10 000s) | `x_train[i]` |
| \(y^{(i)}\) | \(i\)-th restaurant profit (in $10 000s) | `y_train[i]` |
| \(m\) | number of training examples | `m = x_train.shape[0]` |
| \(f_{w,b}(x)\) | linear model prediction | `w * x + b` |
| \(J(w,b)\) | mean-squared-error cost | `compute_cost(...)` |
| \(\alpha\) | learning rate | `alpha` |
| \(\frac{\partial J}{\partial w}\), \(\frac{\partial J}{\partial b}\) | partial derivatives | `dj_dw`, `dj_db` |


## Quick Cheat-Sheet (keep this open while working the skeleton)

### 1. Model
\[
f_{w,b}(x) = w x + b
\]

### 2. Cost (Mean Squared Error, factor ½ for convenience)
\[
J(w,b) = \frac{1}{2m}\sum_{i=0}^{m-1}\bigl(f_{w,b}(x^{(i)}) - y^{(i)}\bigr)^2
\]

**Loop skeleton**
```python
cost_sum = 0
for i in range(m):
    f_wb = w * x[i] + b
    cost_sum += (f_wb - y[i]) ** 2
J = cost_sum / (2 * m)
```

**Vectorized alternate**
```python
f_wb = w * x + b          # broadcasting
J = np.mean((f_wb - y) ** 2) / 2
```

### 3. Gradients
\[
\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=0}^{m-1}(f_{w,b}(x^{(i)}) - y^{(i)}) x^{(i)}
\]
\[
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=0}^{m-1}(f_{w,b}(x^{(i)}) - y^{(i)})
\]

### 4. Gradient-descent update
\[
w \leftarrow w - \alpha \frac{\partial J}{\partial w},\qquad
b \leftarrow b - \alpha \frac{\partial J}{\partial b}
\]

### 5. Typical hyper-parameters for this data set
- `alpha = 0.01`
- `iterations = 1500`
- start from \(w=0\), \(b=0\)

### 6. Expected final parameters (reference)
- \(w \approx 1.166\), \(b \approx -3.630\)
- Profit for pop. 35 000 ≈ $4 520
- Profit for pop. 70 000 ≈ $45 342

### 7. Audience quick tips
| Audience | What they care about | How to present |
|----------|----------------------|----------------|
| CEO / Executive | “Which cities look profitable?” | One clean scatter + fitted line, two concrete $ predictions, no formulas |
| Data Scientist / Analyst | Correctness of \(J\), convergence, residual diagnostics | Cost-history plot, vectorized code, gradient checks |
| Mixed / Technician | Both insight and implementation | Headings that let each reader skip sections; appendix for math |


## Workflow Flowchart (desired outcome)

```
[Load population & profit data]
          |
          v
[Explore: shapes, scatter plot]
          |
          v
[Implement compute_cost (loop + vectorized)]
          |
          v
[Implement compute_gradient]
          |
          v
[Run batch gradient descent (α, iters)]
          |
          v
[Plot fitted line + cost history]
          |
          v
[Predict profits for candidate cities]
          |
          v
[Simulation: vary α / init / noise → new results]
          |
          v
[Audience-adapted summary (exec vs analyst)]
```


## 0. Packages

We use only NumPy and Matplotlib (plus a few standard-library helpers). No external ML libraries – everything is built from first principles.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
print("Libraries imported.")


## 1. Problem Statement

Suppose you are the **CEO of a restaurant franchise** considering new cities for expansion.
- You already have restaurants in many cities and possess historical data: **city population** and **average monthly profit**.
- Negative profit values indicate a loss.
- Candidate cities have known populations; you want a data-driven estimate of expected profit.

**Business question:** Which candidate cities are likely to generate higher profits?

We will answer it by fitting a linear model
\[
f_{w,b}(x) = w x + b
\]
where \(x\) is population (in units of 10 000 people) and the prediction is profit (in units of $10 000).


## 2. Dataset

The classic data set contains 97 examples.  
We load it directly from the project `data/` folder (no external `utils` module required).


In [ ]:
# Load the classic population–profit data
data_path = Path("data/ex1data1.txt")
if not data_path.exists():
    # fallback for different working directories
    data_path = Path("/home/workdir/artifacts/data/ex1data1.txt")

data = np.loadtxt(data_path, delimiter=",")
x_train = data[:, 0]   # population in 10 000s
y_train = data[:, 1]   # profit in $10 000s
m = x_train.shape[0]

print(f"Type of x_train: {type(x_train)}")
print(f"First five populations: {x_train[:5]}")
print(f"First five profits:     {y_train[:5]}")
print(f"Shape of x_train: {x_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Number of training examples m = {m}")


### Visualise the raw data


In [ ]:
plt.figure()
plt.scatter(x_train, y_train, marker="x", c="r", s=40, label="Training data")
plt.title("Profits vs. Population per City")
plt.ylabel("Profit in $10,000s")
plt.xlabel("Population of City in 10,000s")
plt.legend()
plt.tight_layout()
plt.savefig("/home/workdir/artifacts/restaurant_profits_scatter.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: restaurant_profits_scatter.png")


## 3. Compute Cost – Exercise 1 (Solution)

### Loop implementation (pedagogical)


In [ ]:
def compute_cost(x, y, w, b):
    """
    Computes the cost function for linear regression (explicit loop).
    Args:
        x (ndarray): Shape (m,) – city populations
        y (ndarray): Shape (m,) – profits
        w, b (scalar): model parameters
    Returns:
        total_cost (float)
    """
    m = x.shape[0]
    cost_sum = 0.0
    for i in range(m):
        f_wb = w * x[i] + b
        cost = (f_wb - y[i]) ** 2
        cost_sum += cost
    total_cost = (1 / (2 * m)) * cost_sum
    return total_cost

# Quick sanity checks (should match the classic values)
print("Cost at (w,b)=(0,0):     ", round(compute_cost(x_train, y_train, 0, 0), 6))
print("Cost at (w,b)=(0.2,0.2): ", round(compute_cost(x_train, y_train, 0.2, 0.2), 6))


### Alternate – fully vectorized cost


In [ ]:
def compute_cost_vectorized(x, y, w, b):
    """Vectorized cost using NumPy broadcasting."""
    f_wb = w * x + b
    return np.mean((f_wb - y) ** 2) / 2

print("Vectorized cost (0,0):    ", round(compute_cost_vectorized(x_train, y_train, 0, 0), 6))
print("Vectorized cost (0.2,0.2):", round(compute_cost_vectorized(x_train, y_train, 0.2, 0.2), 6))
assert abs(compute_cost(x_train, y_train, 0, 0) - compute_cost_vectorized(x_train, y_train, 0, 0)) < 1e-9
print("Loop and vectorized implementations agree.")


## 4. Compute Gradient – Exercise 2 (Solution)


In [ ]:
def compute_gradient(x, y, w, b):
    """
    Computes the gradient of the cost w.r.t. w and b (explicit loop).
    Returns:
        dj_dw, dj_db (scalars)
    """
    m = x.shape[0]
    dj_dw = 0.0
    dj_db = 0.0
    for i in range(m):
        f_wb = w * x[i] + b
        err = f_wb - y[i]
        dj_dw += err * x[i]
        dj_db += err
    dj_dw /= m
    dj_db /= m
    return dj_dw, dj_db

print("Gradient at (0,0):    ", compute_gradient(x_train, y_train, 0, 0))
print("Gradient at (0.2,0.2):", compute_gradient(x_train, y_train, 0.2, 0.2))


### Alternate – vectorized gradient


In [ ]:
def compute_gradient_vectorized(x, y, w, b):
    f_wb = w * x + b
    err = f_wb - y
    dj_dw = np.mean(err * x)
    dj_db = np.mean(err)
    return dj_dw, dj_db

print("Vectorized grad (0,0):    ", compute_gradient_vectorized(x_train, y_train, 0, 0))
print("Vectorized grad (0.2,0.2):", compute_gradient_vectorized(x_train, y_train, 0.2, 0.2))


## 5. Learning parameters with batch gradient descent


In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent.
    Returns updated w, b and the cost history (for diagnostics).
    """
    w = copy.deepcopy(w_in)
    b = b_in
    J_history = []
    for i in range(num_iters):
        dj_dw, dj_db = gradient_function(x, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        if i < 100000:
            J_history.append(cost_function(x, y, w, b))
        if i % math.ceil(num_iters / 10) == 0 or i == num_iters - 1:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}")
    return w, b, J_history

# Run with the classic hyper-parameters
initial_w = 0.0
initial_b = 0.0
iterations = 1500
alpha = 0.01

w_final, b_final, J_hist = gradient_descent(
    x_train, y_train, initial_w, initial_b,
    compute_cost, compute_gradient, alpha, iterations
)
print(f"\nw, b found by gradient descent: {w_final:.6f}, {b_final:.6f}")


### Plot the linear fit and cost history


In [ ]:
# Predictions on the training set
predicted = w_final * x_train + b_final

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: data + fitted line
axes[0].scatter(x_train, y_train, marker="x", c="r", label="Training data")
axes[0].plot(x_train, predicted, c="b", label=f"Fit: f = {w_final:.3f}x + ({b_final:.3f})")
axes[0].set_title("Profits vs. Population (with linear fit)")
axes[0].set_xlabel("Population of City in 10,000s")
axes[0].set_ylabel("Profit in $10,000s")
axes[0].legend()

# Right: cost history
axes[1].plot(J_hist, c="g")
axes[1].set_title("Cost J(w,b) during gradient descent")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Cost")
axes[1].set_yscale("log")

plt.tight_layout()
plt.savefig("/home/workdir/artifacts/restaurant_profits_fitted.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: restaurant_profits_fitted.png")


### Business predictions for candidate cities


In [ ]:
# Population of 35 000 → model input 3.5
# Population of 70 000 → model input 7.0
predict1 = 3.5 * w_final + b_final
predict2 = 7.0 * w_final + b_final
print(f"For population = 35,000, we predict a profit of ${predict1*10000:,.2f}")
print(f"For population = 70,000, we predict a profit of ${predict2*10000:,.2f}")


## 6. More Practice

1. **Residual analysis** – Compute the residuals \(y - f_{w,b}(x)\). What is the mean residual? Plot a residual histogram.
2. **R²** – Calculate the coefficient of determination for the fitted model.
3. **Closed-form solution** – Using the normal equation (or `np.polyfit`), recover the same \(w,b\). Compare to the GD result.
4. **Feature scaling experiment** – Suppose you accidentally scaled population by 1000 instead of 10 000. How does that affect the optimal learning rate?


In [ ]:
# 1. Residuals
residuals = y_train - predicted
print(f"Mean residual: {np.mean(residuals):.6f}  (should be near zero)")
print(f"Std of residuals: {np.std(residuals):.4f}")

# 2. R²
ss_res = np.sum(residuals ** 2)
ss_tot = np.sum((y_train - np.mean(y_train)) ** 2)
r2 = 1 - ss_res / ss_tot
print(f"R² = {r2:.4f}")

# 3. Closed-form (polyfit degree 1)
w_cf, b_cf = np.polyfit(x_train, y_train, 1)
print(f"Closed-form: w = {w_cf:.6f}, b = {b_cf:.6f}")
print(f"GD vs closed-form difference: Δw = {abs(w_final-w_cf):.2e}, Δb = {abs(b_final-b_cf):.2e}")


## 7. Simulation Section – Explore Hyper-parameters & Robustness

Change the values in the cell below and re-run to observe how the final cost, parameters and predictions change.


In [ ]:
# ========== SIMULATION CONTROLS (edit these) ==========
SIM_ALPHA        = 0.01      # try 0.001, 0.03, 0.1 …
SIM_ITERS        = 1500      # try 300, 5000
SIM_W0, SIM_B0   = 0.0, 0.0  # try different initialisations
NOISE_STD        = 0.0       # add Gaussian noise to y (try 0.5, 1.0)
RANDOM_SEED      = 42
# =====================================================

rng = np.random.default_rng(RANDOM_SEED)
y_sim = y_train + rng.normal(0, NOISE_STD, size=m)

w_sim, b_sim, J_sim = gradient_descent(
    x_train, y_sim, SIM_W0, SIM_B0,
    compute_cost, compute_gradient, SIM_ALPHA, SIM_ITERS
)

print(f"Simulated (w, b) = ({w_sim:.4f}, {b_sim:.4f})")
print(f"Final cost       = {J_sim[-1]:.4f}")
print(f"Profit @ 35k     = ${ (3.5*w_sim + b_sim)*10000 :,.0f}")
print(f"Profit @ 70k     = ${ (7.0*w_sim + b_sim)*10000 :,.0f}")

# Quick cost-curve comparison
plt.figure(figsize=(8, 4))
plt.plot(J_hist, label="Original (α=0.01, no noise)")
plt.plot(J_sim,  label=f"Sim (α={SIM_ALPHA}, noise={NOISE_STD})")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Cost trajectories under different settings")
plt.legend()
plt.tight_layout()
plt.savefig("/home/workdir/artifacts/restaurant_profits_simulation.png", dpi=120, bbox_inches="tight")
plt.show()


## 8. Audience Adaptation Notes

(Based on the supplied documents *What to Consider When Considering the Audience* and *Audience and Situation Analysis*.)

### Data-literacy dimension
- **Highly data-literate** (data scientists, quantitative analysts): show the cost surface / contour, residual diagnostics, vectorized vs loop comparison, R², closed-form verification.
- **Low data-literacy** (many executives, field managers): avoid the symbols \(J\), \(\alpha\), partial derivatives. Speak only of “how well the straight line fits the historical cities” and the two concrete dollar predictions.

### Subject-knowledge dimension
- Restaurant / franchise executives already understand “population drives foot traffic → profit”. No need to explain the business logic; jump straight to the numbers.
- An audience from a completely different industry needs a one-sentence framing: “Larger cities tend to produce higher monthly profits for this restaurant concept; we quantified that relationship.”

### Presentation formats by audience type
| Audience type | Recommended artefacts |
|---------------|-----------------------|
| **Executive / CEO** (primary client) | One scatter+fit plot, two bullet-point $ forecasts, a single sentence recommendation. Put the math in an appendix. |
| **Technical supervisor / peer data scientist** | Full cost history, gradient checks, R², residual plot, both loop & vectorized code, simulation sensitivity. |
| **Mixed audience** | Clear section headings so each reader can skim; executive summary first, technical details later. |

### Writing style reminder (from the data-analysis-report structure)
- Make the writing “invisible” – no flowery language, no process digressions.
- Organise around the conversation you want to have with the CEO: “Here is the historic relationship, here is what it implies for the two candidate cities.”


## 9. Re-usable Template Pattern

For any new single-variable linear-regression problem follow the same skeleton:

1. Load / create `x_train`, `y_train`
2. Scatter-plot & inspect \(m\)
3. Implement `compute_cost` (loop + vectorized)
4. Implement `compute_gradient`
5. Call `gradient_descent` with sensible \(\alpha\) and iterations
6. Plot fit + cost history
7. Make business predictions
8. Add residual / R² diagnostics
9. Run a short simulation varying \(\alpha\), initialisation or noise
10. Write two short narratives: one for the executive, one for the technical reader

Save this notebook as a starting point and swap the data domain.


---
**End of Extended Solution**  
You now have a complete, production-ready implementation of univariate linear regression built from first principles, together with diagnostics, simulation controls and audience-aware communication guidance.
